# BITS Pilani - WILP | M.Tech (AIML)
## Machine Learning (S1-26_AIMLZG565) - Assignment 2
**Student Name:** Lokesh Panda  
**Student ID:** 2026AK05158  

---
### Project: Email Spam Classification System using UCI Spambase
This notebook demonstrates the end-to-end machine learning pipeline:
1. Loading the UCI Spambase dataset (57 continuous features, 4,601 instances)
2. Exploratory analysis and stratified train/test split (80/20)
3. Feature standardization using `StandardScaler`
4. Training and evaluating 5 classification models:
   - Logistic Regression
   - Decision Tree Classifier
   - K-Nearest Neighbor (KNN)
   - Naive Bayes Classifier (GaussianNB)
   - Support Vector Machine (SVM with RBF kernel)
5. Computing 6 evaluation metrics: Accuracy, AUC, Precision, Recall, F1, and MCC
6. Model artifact persistence for interactive Streamlit deployment

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve
)

### 1. Dataset Loading and Preprocessing

In [2]:
data = fetch_openml(name='spambase', version=1, as_frame=True)
df = data.frame.copy()
df['class'] = df['class'].astype(int)

# Normalize column names
clean_cols = [c.replace(';', 'semicolon').replace('(', 'round_bracket').replace('[', 'square_bracket').replace('!', 'exclamation').replace('$', 'dollar').replace('#', 'hash') for c in df.columns]
df.columns = clean_cols

print(f"Total instances: {df.shape[0]}, Features: {df.shape[1] - 1}")
print(df['class'].value_counts().rename({0: 'Non-Spam (0)', 1: 'Spam (1)'}))

### 2. Stratified Train / Test Partitioning and Scaling

In [3]:
X = df.drop(columns=['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

### 3. Model Training and Comprehensive Evaluation

In [4]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, random_state=42), False),
    'KNN': (KNeighborsClassifier(n_neighbors=5), True),
    'Naive Bayes': (GaussianNB(), False),
    'Support Vector Machine': (SVC(kernel='rbf', probability=True, random_state=42), True)
}

records = []
for name, (clf, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    
    records.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })

comparison_table = pd.DataFrame(records)
comparison_table